# 04e search sft postration nq

**Arm:** SFT arm 1 - Search-7B-SFT - model: `Jianwen/Search-7B-SFT`

**Before running:** Settings -> Internet: **ON**, Accelerator: **GPU**. Input dataset
`mehedikhan72/rag-faith-hagrid-adv-noplant` must be attached (handled by kernel-metadata).

Step 5's re-inference pass. Step 4 asked each model to answer questions over clean top-5
evidence, one citation per sentence. `03_adversarial_construction.py` then took each cited
sentence, lifted a 4-word span from it, and planted that span into a passage the model had
*not* cited for that sentence -- in three flavours (a random unrelated passage, a
retrieved-but-uncited passage, a passage cited for a different sentence of the same
answer).

This notebook re-runs the *same* model on those doctored contexts with the *same* prompt
and records what it says. Scoring happens elsewhere.

All four notebooks are generated by `scripts/generate_postration_notebooks.py`. Edit that
script, not this file.


---

**W2 NO-PLANTING CONTROL RUN.** This notebook is the control twin of
`04x_postration_*`: the attached dataset is the `-noplant` one, whose doctored slot has had
the appended span removed and nothing else changed. Every code cell below is byte-identical
to the planted notebook except `OUT_DIR`. The quantity of interest is the difference between
the two conditions, not either rate alone.

## Stage 0 - Install dependencies

Identical to the 02x generation notebooks, for the same empirically-found reasons: Kaggle
randomly assigns a T4 (`sm_75`) or a P100 (`sm_60`), and its default preinstalled torch
has dropped Pascal support, so a P100 session fails at `generate()` with `CUDA error: no
kernel image is available for execution on the device`. v1 pinned `torch==2.4.1+cu121`;
v2 keeps that pin and runs on v1's own docker image, pinned in kernel-metadata (see the 02x
generator). `torchvision`/`torchaudio` are removed because their ABI mismatch against the
downgraded torch breaks the lazy import of `Qwen2ForCausalLM`.

In [ ]:
import subprocess

subprocess.run(
    ["pip", "install", "-q", "torch==2.4.1", "--index-url",
     "https://download.pytorch.org/whl/cu121"],
    check=True,
)
subprocess.run(["pip", "uninstall", "-y", "-q", "torchvision", "torchaudio"], check=True)
subprocess.run(["pip", "install", "-q", "transformers>=4.46", "accelerate"], check=True)

import sys
import torch
import transformers
print(f"python {sys.version.split()[0]}, torch {torch.__version__}, "
      f"transformers {transformers.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  device {i}: {torch.cuda.get_device_name(i)}, "
              f"capability {torch.cuda.get_device_capability(i)}")
print("Installed.")

## Stage 1 - Config (the only cell that differs between the 4 arms)

In [ ]:
import os
from pathlib import Path


def find_input_dir(*required):
    """Locate an attached Kaggle dataset by the files it contains.

    Failing loudly here costs seconds; a wrong path costs the whole GPU session.
    """
    for dirpath, _dirnames, filenames in os.walk("/kaggle/input"):
        if set(required) <= set(filenames):
            return Path(dirpath)
    listing = "\n".join(
        os.path.join(dp, f)
        for dp, _dn, fs in os.walk("/kaggle/input") for f in fs
    ) or "  (/kaggle/input is empty)"
    raise SystemExit(
        f"could not find {required} under /kaggle/input. "
        f"Is the dataset attached in kernel-metadata dataset_sources?\n"
        f"Tree:\n{listing}"
    )

# ---- The only block that differs across the 4 post-rationalization notebooks ----
MODEL_NAME = "Jianwen/Search-7B-SFT"
ARM_KEY = "search_sft"
ARM_LABEL = "SFT arm 1 - Search-7B-SFT"
# ---------------------------------------------------------------------------------

DATASET_DIR = find_input_dir(f"search_sft__rel_uncited.json")
OUT_DIR = Path("/kaggle/working/postration_noplant_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Set to an int (e.g. 3) to smoke-test each variant; None runs every record.
N_LIMIT = None

print(f"Adversarial data: {DATASET_DIR}")
print(f"Model: {MODEL_NAME}")
print(f"Arm: {ARM_KEY} ({ARM_LABEL})")
print(f"N_LIMIT: {N_LIMIT}")

## Stage 2 - Load the adversarial contexts

**No retrieval happens in this notebook.** The adversarial JSON files already carry a
fully-built, doctored 5-chunk context per record (`doctored_chunks`), produced locally by
`03_adversarial_construction.py`. Re-running BM25 here would overwrite the doctoring and
silently invalidate the test.

Each context is the original top-5 with exactly one slot replaced by a chunk that has had
a 4-word span appended -- the span taken from the sentence the model itself attributed to
a different passage. The genuine cited chunk is still present in every variant, so the
model always has a legitimate source available; the question is whether it *also* reaches
for the planted one.

In [ ]:
import json
from pathlib import Path

VARIANTS = ['random']


def load_json(path):
    with open(path) as f:
        return json.load(f)


def write_json(path, rows, indent=2):
    with open(path, "w") as f:
        json.dump(rows, f, indent=indent)
        f.write("\n")


datasets = {}
for variant in VARIANTS:
    path = DATASET_DIR / f"{ARM_KEY}__{variant}.json"
    datasets[variant] = load_json(path)
    print(f"{variant}: {len(datasets[variant])} records  <- {path.name}")

total = sum(len(v) for v in datasets.values())
print(f"\nTotal generations to run for this arm: {total}")

## Stage 3 - Prompt building (byte-identical to Step 4's `build_messages`)

This **must** match the 02x prompt exactly, including the strict one-citation-per-sentence
instructions. The measurement compares citation behaviour on doctored evidence against
citation behaviour on clean evidence; if the prompt drifted between the two runs, any
difference found would be uninterpretable. Nothing here hints that a passage was tampered
with.

In [ ]:
import re

CITATION_RE = re.compile(r"\[(\d+)\]")


def build_messages(question, retrieved_chunks):
    context_block = "\n\n".join(
        f"[{i + 1}] {c['text']}" for i, c in enumerate(retrieved_chunks)
    )
    user_msg = (
        "Answer the question using ONLY the numbered passages below.\n\n"
        "Format your answer as separate sentences. Each sentence must end with exactly "
        "one citation in square brackets, like this:\n"
        "First sentence here [2]. Second sentence here [4].\n\n"
        "Rules:\n"
        "- Exactly one citation per sentence. Never [1][2].\n"
        "- The citation goes at the end of the sentence, before the period.\n"
        "- Do not add any text outside these sentences.\n"
        "- If the passages do not contain the answer, reply exactly: I don't know\n\n"
        f"Passages:\n{context_block}\n\n"
        f"Question: {question}"
    )
    return [{"role": "user", "content": user_msg}]


def parse_citation_markers(generated_text, n_chunks):
    markers = sorted({int(m) for m in CITATION_RE.findall(generated_text)})
    return [m for m in markers if 1 <= m <= n_chunks]

## Stage 4 - Load model

**`device_map={"": 0}`, not `"auto"`** - same finding as the generation step: on a single
16GB card, accelerate's automatic estimation conservatively offloaded part of the model to
CPU (`hf_device_map` showing `{0, 'cpu'}`), costing ~4.75x in generation time. A 7B model
in fp16 fits with room for the KV cache at this prompt length.

v2: on T4x2, the only GPU Kaggle now offers, the layers are split across both cards; see the
02x generator.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# v2: Kaggle's only GPU is now T4x2 (14.6 GiB each), too small for 7B fp16 on one card, so
# layers are split evenly across both cards. max_memory leaves no room on CPU, so the
# assert below catches any offload. On a single card this is the v1 placement unchanged.
if torch.cuda.device_count() > 1:
    placement = dict(device_map="balanced",
                     max_memory={i: "13GiB" for i in range(torch.cuda.device_count())})
else:
    placement = dict(device_map={"": 0})
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    **placement,
)
model.eval()
placed = set(getattr(model, "hf_device_map", {"": 0}).values())
assert not placed & {"cpu", "disk"}, f"model offloaded off-GPU: {placed}"
print(f"Loaded {MODEL_NAME} on devices: {sorted(map(str, placed))}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB / "
      f"reserved: {torch.cuda.memory_reserved() / 1e9:.1f} GB")

## Stage 5 - Re-generate on doctored contexts, persist incrementally

Greedy decoding (`do_sample=False`), same as the clean run, so the only variable versus
that run is the doctored passage.

**No metrics are computed here, deliberately.** Each row records the question, the claim
and span that were planted, which slot was doctored, the new answer, and the citation
markers it contains. Whether the citation actually moved to the planted passage is a
judgement call (LLM judge or human annotation) made downstream -- baking a boolean in here
would lock in a scoring rule before it has been agreed.

In [ ]:
import time

t0 = time.time()
n_done = 0

for variant in VARIANTS:
    records = datasets[variant]
    if N_LIMIT:
        records = records[:N_LIMIT]
    out_path = OUT_DIR / f"postration_{ARM_KEY}_{variant}.json"
    written = []

    for rec in records:
        doctored_chunks = rec["doctored_chunks"]
        messages = build_messages(rec["question"], doctored_chunks)
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=False,
            )
        new_answer = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )

        written.append({
            "question_id": rec["question_id"],
            "question": rec["question"],
            "arm": ARM_KEY,
            "variant": variant,
            "claim_text": rec["claim_text"],
            "claim_marker": rec["claim_marker"],
            "extracted_span": rec["extracted_span"],
            "cited_chunk_id": rec["cited_chunk_id"],
            "substituted_slot_index": rec["substituted_slot_index"],
            "doctored_marker": rec["substituted_slot_index"] + 1,
            "doctored_source_chunk_id": rec["doctored_source_chunk_id"],
            "new_answer": new_answer,
            "new_citation_markers": parse_citation_markers(new_answer, len(doctored_chunks)),
        })

        # A JSON array can't be appended to line by line the way JSONL could, so rewrite
        # the whole file every 20 records. Same purpose as the old per-record flush: a
        # Kaggle session dying mid-run must not cost the whole batch.
        n_done += 1
        if n_done % 20 == 0:
            write_json(out_path, written)
            print(f"  {n_done} generations done ({time.time() - t0:.0f}s elapsed)")

    write_json(out_path, written)
    print(f"[{variant}] wrote {len(written)} records -> {out_path.name}")

print(f"\nAll done: {n_done} generations in {time.time() - t0:.0f}s")

## Stage 6 - Sanity read + persist

Counts only, no rates. This exists so an obviously broken run (empty answers, no markers
at all, wrong record count) is visible before pulling results down.

In [ ]:
import shutil

print(f"=== {ARM_LABEL} ({ARM_KEY}) ===")
print(f"{'variant':<16} {'rows':>6} {'no markers':>12} {'mean markers':>14}")
for variant in VARIANTS:
    path = OUT_DIR / f"postration_{ARM_KEY}_{variant}.json"
    if not path.exists():
        continue
    rows = load_json(path)
    empty = sum(1 for r in rows if not r["new_citation_markers"])
    mean = sum(len(r["new_citation_markers"]) for r in rows) / max(len(rows), 1)
    print(f"{variant:<16} {len(rows):>6} {empty:>12} {mean:>14.2f}")

print("\n--- 2 sample rows ---")
for r in load_json(OUT_DIR / f"postration_{ARM_KEY}_{VARIANTS[0]}.json")[:2]:
    print(f"Q: {r['question']}")
    print(f"planted span: {r['extracted_span']!r} into slot [{r['doctored_marker']}]")
    print(f"new answer: {r['new_answer']}")
    print(f"markers: {r['new_citation_markers']}\n")

shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print(f"Zipped to {OUT_DIR}.zip")